In [1]:
import Pkg
Pkg.activate("../../.")

  Activating project at `~/dev/MyCloudAtlas.jl`


# Traveling-Wave Discovery (Re=300)

Goal: find novel traveling-wave solutions by fuzzing TWModel seeds, deduplicating by
wave speeds + norms + shear, and then promoting unique candidates to a full Channelflow
`findsoln` run.

This notebook is designed to handle multiple symmetry groups and discretizations.

Notes:
- The default guess strategy is random (Gaussian coefficients rescaled to `xnorm`,
  with random cx/cz if those speeds are kept).
- `:shear_target` uses rejection sampling until wall-shear is within `shear_tol`
  of `shear_target`.
- `:shear_band` (alias `:shear_modes`) samples dominant shear modes to land in a
  shear window [`shear_min`, `shear_max`] while keeping the remaining modes small.
- The `findsoln` stage will write results to `out_dir`.

In [2]:
using CloudAtlas
using LinearAlgebra
using Statistics
using Random
using Dates
using DelimitedFiles
using Serialization
using Base.Threads
using ChannelflowWrapper

## Configuration

In [3]:
# Domain sizes (α = 2π/Lx, γ = 2π/Lz)
# Choose these to match the target Channelflow DNS box.
α, γ = 2π/6.0, 2π/4.0

Re = 300.0

# Discretizations: (J, K, L)
# Use a ladder: start coarse, then project/refine to finer discretizations.
discretization_ladder = [
    (1, 3, 5),
    # (2, 4, 7),
    # (2, 4, 7),
    # (3, 5, 9),
]

# Symmetry groups to explore
sx, sy, sz, tx, tz = CloudAtlas.halfbox_symmetries()

symmetry_groups = [
    (
        name = "sxytxz",
        H = [(sx * sy) * (tx * tz)],
        symm_file = joinpath(@__DIR__, "sxytxz.asc"),
    ),
    (
        name = "sztx",
        H = [sz * tx],
        symm_file = joinpath(@__DIR__, "sztx.asc"),
    ),
    (
        name = "tx",
        H = [tx],
        symm_file = joinpath(@__DIR__, "tx.asc"),
    ),
    (
        name = "tz",
        H = [tz],
        symm_file = joinpath(@__DIR__, "tz.asc"),
    ),
    (
        name = "txtz",
        H = [tx * tz],
        symm_file = joinpath(@__DIR__, "txtz.asc"),
    ),
    (
        name = "sx_tx",
        H = [sx * tx],
        symm_file = joinpath(@__DIR__, "sxtx.asc"),
    ),
    (
        name = "sztx",
        H = [sz * tx],
        symm_file = joinpath(@__DIR__, "sztx.asc"),
    ),
    (
        name = "sxtz",
        H = [sx * tz],
        symm_file = joinpath(@__DIR__, "sxtz.asc"),
    ),
    (
        name = "sztz",
        H = [sz * tz],
        symm_file = joinpath(@__DIR__, "sztz.asc"),
    ),
]

# Attempts per symmetry at the coarsest ladder level
attempts_per_level = 1000

# Only run promotion at the highest ladder level
promote_each_level = true

# Hookstep parameters
hookparams = SearchParams(
    ftol = 1e-8,
    xtol = 1e-10,
    δ = 0.02,
    Nnewton = 20,
    Nhook = 4,
    Nmusearch = 6,
    verbosity = 0,
)

# Guess strategy options: :random, :shear_target, :shear_band (aka :shear_modes)
guess_strategy = :shear_band
xnorm = 0.4
shear_target = 1.2
shear_tol = 0.05
shear_min = 1.0
shear_max = 3.0

# Dedup tolerances
fp_tol = (
    cx = 1e-3,
    cz = 1e-3,
    nm = 2e-2,
    shear = 2e-2,
)

# Accept/reject thresholds
norm_threshold = 1e-3
speed_threshold = 1e-5
promote_norm_threshold = 1e-2
promote_residual_tol = 1e-6

# Channelflow promotion settings
T = 10.0
out_dir = joinpath(@__DIR__, "tw_discovery_re300")
mkpath(out_dir)

# Reference field for Channelflow conversions
reference_path = joinpath(@__DIR__, "TW1-2pi1piRe200-40x49x40.nc")
reference_field_converted = joinpath(out_dir, "reference_field_$(α)_$(γ).nc")

"/home/ebenq/dev/MyCloudAtlas.jl/notebooks/tw_discovery/tw_discovery_re300/reference_field_1.0471975511965976_1.5707963267948966.nc"

## Helper types and functions

In [4]:
function save_summary(path, solutions, model)
    header = ["id" "cx" "cz" "norm" "shear"]
    if isempty(solutions)
        writedlm(path, header, ',')
        return
    end

    rows = Matrix{Float64}(undef, length(solutions), 5)
    for (i, ξ) in enumerate(solutions)
        x, cx, cz = extract_components(ξ, model)
        rows[i, 1] = i
        rows[i, 2] = cx
        rows[i, 3] = cz
        rows[i, 4] = norm(x)
        rows[i, 5] = shear(x, model)
    end

    writedlm(path, vcat(header, rows), ',')
end

function project_solution(ξ_from, model_from::ODEModel, model_to::ODEModel)
    x_from, cx_from, cz_from = extract_components(ξ_from, model_from)
    x_to = changebasis(x_from, model_from.ijkl, model_to.ijkl)
    cx_to = model_to.keep_cx ? cx_from : zero(cx_from)
    cz_to = model_to.keep_cz ? cz_from : zero(cz_from)
    return state_to_xi(model_to, ODEState(x_to, cx_to, cz_to))
end

function refine_projected_solutions(model_from::ODEModel, model_to::ODEModel, Re;
    solutions_from,
    hookparams,
    norm_threshold = 1e-3,
    speed_threshold = 1e-5,
)
    refined = Vector{Vector{Float64}}()
    fingerprints = Vector{SolutionFingerprint}()
    io_lock = ReentrantLock()

    @threads for i in eachindex(solutions_from)
        ξ_guess = project_solution(solutions_from[i], model_from, model_to)
        ξ_star, converged = CloudAtlas.hookstepsolve(model_to, Re, ξ_guess, hookparams)
        if converged
            x, cx, cz = extract_components(ξ_star, model_to)
            if norm(x) > norm_threshold && (abs(cx) > speed_threshold || abs(cz) > speed_threshold)
                fp = fingerprint(model_to, ξ_star)
                lock(io_lock) do
                    if is_distinct(fp, fingerprints; tol = fp_tol)
                        push!(fingerprints, fp)
                        push!(refined, ξ_star)
                    end
                end
            end
        end
    end

    return refined, fingerprints
end

refine_projected_solutions (generic function with 1 method)

## Fuzzing + deduplication

In [5]:
function fuzz_tw_solutions(model::TWModel, Re::Real;
    n_attempts = 1000,
    xnorm = 0.4,
    strategy = :random,
    shear_target = 1.2,
    shear_tol = 0.05,
    hookparams = SearchParams(),
    norm_threshold = 1e-3,
    speed_threshold = 1e-5,
)
    m = length(model)
    solutions = Vector{Vector{Float64}}()
    fingerprints = Vector{SolutionFingerprint}()

    data_lock = ReentrantLock()
    io_lock = ReentrantLock()

    rngs = [MersenneTwister(0xC0FFEE + i) for i in 1:Threads.maxthreadid()]
    progress = Threads.Atomic{Int}(0)
    progress_every = max(1, n_attempts ÷ 100)

    @threads for attempt in 1:n_attempts
        tid = threadid()
        rng = tid <= length(rngs) ? rngs[tid] : Random.default_rng()
        ξ_guess = build_guess(model, rng;
            strategy = strategy,
            xnorm = xnorm,
            target = shear_target,
            tol = shear_tol,
            shear_min = shear_min,
            shear_max = shear_max,
        )

        ξ_star, converged = CloudAtlas.hookstepsolve(model, Re, ξ_guess, hookparams)

        if converged
            x, cx, cz = extract_components(ξ_star, model)
            if norm(x) > norm_threshold && (abs(cx) > speed_threshold || abs(cz) > speed_threshold)
                fp = fingerprint(model, ξ_star)
                lock(data_lock) do
                    if is_distinct(fp, fingerprints; tol = fp_tol)
                        push!(fingerprints, fp)
                        push!(solutions, ξ_star)
                        lock(io_lock) do
                            println("[Thread $(threadid())] Unique TW: cx=$(round(fp.cx, digits=4)), cz=$(round(fp.cz, digits=4)), |x|=$(round(fp.nm, digits=4)), shear=$(round(fp.shear, digits=4))")
                        end
                    end
                end
            end
        end

        done = Threads.atomic_add!(progress, 1)
        if done % progress_every == 0 || done == n_attempts
            lock(io_lock) do
                pct = round(100 * done / n_attempts; digits=1)
                println("Progress: $(done)/$(n_attempts) ($(pct)%) (unique=$(length(solutions)))")
            end
        end
    end

    println("Done. Found $(length(solutions)) unique solutions.")
    return solutions, fingerprints
end

fuzz_tw_solutions (generic function with 1 method)

## Channelflow promotion

In [6]:
function ensure_reference_field(reference_path, reference_field_converted; α, γ)
    if !isfile(reference_field_converted)
        changegrid(reference_path, reference_field_converted; al = α, ga = γ)
    end
    return reference_field_converted
end

function promote_with_findsoln!(solutions, model, Re;
    symm_file,
    out_dir,
    reference_field_converted,
    T = 10.0,
    promote_norm_threshold = 1e-2,
    promote_residual_tol = 1e-6,
)
    m = length(model)
    mkpath(out_dir)
    io_lock = ReentrantLock()
    progress = Threads.Atomic{Int}(0)
    total = length(solutions)
    progress_every = max(1, total ÷ 100)
    @threads for idx in 1:total
        ξ = solutions[idx]
        x, cx, cz = extract_components(ξ, model)
        cx_eff = model.keep_cx ? cx : 0.0
        cz_eff = model.keep_cz ? cz : 0.0
        resnorm = norm(CloudAtlas.residual(model, x, cx_eff, cz_eff, Re))
        if norm(x) < promote_norm_threshold || resnorm > promote_residual_tol
            lock(io_lock) do
                println("Skipping promotion idx=$(idx): ||x||=$(norm(x)), ||res||=$(resnorm), cx=$(cx_eff), cz=$(cz_eff)")
            end
            done = Threads.atomic_add!(progress, 1)
            if done % progress_every == 0 || done == total
                lock(io_lock) do
                    pct = round(100 * done / total; digits=1)
                    println("Promotion progress: $(done)/$(total) ($(pct)%)")
                end
            end
            continue
        end

        timestamp = Dates.format(now(), "MM-DD-HHMMSS")
        sol_dir = joinpath(out_dir, "sol_$(idx)_$(timestamp)")
        mkpath(sol_dir)

        guess_path = joinpath(sol_dir, "u_guess.nc")
        sigma_file = joinpath(sol_dir, "sigma.asc")

        println("[Thread $(threadid())] coeff2field idx=$(idx)")
        coeff2field(x, model.ijkl, reference_field_converted, guess_path; workdir = sol_dir)
        println("[Thread $(threadid())] save_sigma idx=$(idx)")
        save_sigma(model, cx_eff, cz_eff, T, sigma_file)

        try
            lock(io_lock) do
                println("Promoting idx=$(idx): ||x||=$(norm(x)), ||res||=$(resnorm), cx=$(cx_eff), cz=$(cz_eff)")
            end
            # Run findsoln inside its own directory to avoid temp file collisions.
            # Also override temp dirs for Channelflow's temp_* files.
            println("[Thread $(threadid())] findsoln start idx=$(idx)")
            findsoln(guess_path;
                workdir = sol_dir,
                R = Re,
                eqb = true,
                xrel = model.keep_cx,
                zrel = model.keep_cz,
                symms = abspath(symm_file),
                sigma = sigma_file,
                od = sol_dir,
                T = T,
            )
            println("[Thread $(threadid())] findsoln done idx=$(idx)")
        catch e
            lock(io_lock) do
                println("findsoln failed idx=$(idx): $(e)")
            end
        end

        done = Threads.atomic_add!(progress, 1)
        if done % progress_every == 0 || done == total
            lock(io_lock) do
                pct = round(100 * done / total; digits=1)
                println("Promotion progress: $(done)/$(total) ($(pct)%)")
            end
        end
    end
end

promote_with_findsoln! (generic function with 1 method)

## Main sweep

In [7]:
reference_field_converted = ensure_reference_field(reference_path, reference_field_converted; α = α, γ = γ)

for symm in symmetry_groups
    symm_name = symm.name
    H = symm.H
    symm_file = symm.symm_file

    println("\n=== Symmetry: $(symm_name) ===")
    println("symm_file: $(symm_file)")

    prev_model = nothing
    prev_solutions = Vector{Vector{Float64}}()

    for (level_idx, (J, K, L)) in enumerate(discretization_ladder)
        println("\n--- Ladder level $(level_idx): J,K,L = $(J),$(K),$(L) ---")

        model = ODEModel(α, γ, J, K, L, H; normalize = false, tw = true)
        level_dir = joinpath(out_dir, symm_name, "jkl_$(J)_$(K)_$(L)")
        mkpath(level_dir)

        if level_idx == 1
            solutions, _ = fuzz_tw_solutions(
                model,
                Re;
                n_attempts = attempts_per_level,
                xnorm = xnorm,
                strategy = guess_strategy,
                shear_target = shear_target,
                shear_tol = shear_tol,
                hookparams = hookparams,
                norm_threshold = norm_threshold,
                speed_threshold = speed_threshold,
            )
            prev_solutions = solutions
        else
            solutions, _ = refine_projected_solutions(
                prev_model,
                model,
                Re;
                solutions_from = prev_solutions,
                hookparams = hookparams,
                norm_threshold = norm_threshold,
                speed_threshold = speed_threshold,
            )
            prev_solutions = solutions
        end

        # Save summary of unique solutions
        summary_path = joinpath(level_dir, "solutions_summary.csv")
        save_summary(summary_path, prev_solutions, model)

        # Save raw solutions (for later reuse)
        serialized_path = joinpath(level_dir, "solutions.bin")
        open(serialized_path, "w") do io
            serialize(io, prev_solutions)
        end

        if promote_each_level || level_idx == length(discretization_ladder)
            promote_with_findsoln!(
                prev_solutions,
                model,
                Re;
                symm_file = symm_file,
                out_dir = joinpath(level_dir, "findsoln"),
                reference_field_converted = reference_field_converted,
                T = T,
                promote_norm_threshold = promote_norm_threshold,
                promote_residual_tol = promote_residual_tol,
            )
        end

        prev_model = model
    end
end


=== Symmetry: sxytxz ===
symm_file: /home/ebenq/dev/MyCloudAtlas.jl/notebooks/tw_discovery/sxytxz.asc

--- Ladder level 1: J,K,L = 1,3,5 ---
J,K,L,m == 1,3,5,117
(2J+1)(2K+1)(2L+1) + 1 == 232
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 
Making matrices Cx,Cz...
Phase constraints: keep_cx = false, keep_cz = true
Progress: 0/1000 (0.0%) (unique=0)
Progress: 10/1000 (1.0%) (unique=0)
Progress: 20/1000 (2.0%) (unique=0)
Progress: 30/1000 (3.0%) (unique=0)
Progress: 40/1000 (4.0%) (unique=0)
Progress: 50/1000 (5.0%) (unique=0)
Progress: 60/1000 (6.0%) (unique=0)
Progress: 70/1000 (7.0%) (unique=0)
Progress: 80/1000 (8.0

Excessive output truncated after 524425 bytes.

 res == 0.000308025
GMRES converged. Breaking.
------------------------------------------------
Beginning hookstep calculations.
-------------------------------------------
Hookstep number 0
delta == 0.000549547
Newton step is outside trust region: 
L2Norm(dxN) == 0.000623991 > 0.000549547 == delta
Calculate hookstep dxH(mu) with radius L2Norm(dxH(mu)) == delta
mu, L2Norm(dxH(mu)) == 0, 2.01309
mu, L2Norm(dxH(mu)) == 0.00951342, 0.00110364


## Notes

- To change the guess strategy, set `guess_strategy = :shear_target` or
  `guess_strategy = :shear_band` (alias `:shear_modes`), or add your own generator in
  `build_guess` (see `src/Guessing.jl`).
- If `findsoln` is too heavy, comment out the `promote_with_findsoln!` call and run
  it later using the saved `solutions.bin`.